In [0]:
import json
from pyspark.sql.functions import col, avg, count, round, desc

# 1. Cargar Credenciales
try:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)
    aws_access_key = config['aws_access_key']
    aws_secret_key = config['aws_secret_key']
    bucket_name = "bronce-scrap-date" # Aseguramos el nombre correcto
    print("✅ Credenciales cargadas.")
except:
    print("❌ Error cargando secretos.")

# 2. Leer la Capa Silver (Tu tabla maestra)
ruta_silver = f"s3a://{bucket_name}/silver/master_inmuebles/"
print(f"📖 Leyendo Silver desde: {ruta_silver}")

df_silver = spark.read.format("delta") \
    .option("fs.s3a.access.key", aws_access_key) \
    .option("fs.s3a.secret.key", aws_secret_key) \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .load(ruta_silver)

# 3. Calcular KPI: Precio por Metro Cuadrado
# Filtramos basura y calculamos el ratio
df_gold_kpi = df_silver \
    .filter((col("precio_num").isNotNull()) & (col("area_m2") > 10)) \
    .withColumn("precio_m2", round(col("precio_num") / col("area_m2"), 0))

print("✅ KPI Calculado. Muestra de datos:")
# CAMBIO AQUÍ: Usamos 'barrio' en lugar de 'ubicacion_raw'
display(df_gold_kpi.select("titulo", "barrio", "precio_num", "area_m2", "precio_m2").limit(5))

# 4. Agrupación por BARRIO (Aggregation)
df_mercado_resumen = df_gold_kpi.groupBy("barrio") \
    .agg(
        count("id_original").alias("total_ofertas"),
        round(avg("precio_num"), 0).alias("precio_promedio"),
        round(avg("area_m2"), 0).alias("area_promedio"),
        round(avg("precio_m2"), 0).alias("valor_m2_promedio")
    ) \
    .orderBy(desc("valor_m2_promedio"))

# Filtramos barrios con muy pocos datos para evitar ruido
df_mercado_top = df_mercado_resumen.filter(col("total_ofertas") >= 1)

print("🏆 Top Barrios por Valor del m²:")
display(df_mercado_top)

# 5. Guardar Resumen Gold (Delta)
ruta_gold_resumen = f"s3a://{bucket_name}/gold/mercado_resumen/"

df_mercado_top.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("fs.s3a.access.key", aws_access_key) \
    .option("fs.s3a.secret.key", aws_secret_key) \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .save(ruta_gold_resumen)
    
print(f"💾 Resumen Gold guardado en: {ruta_gold_resumen}")

# 6. GUARDAR PARA LA APP (Gold Detail - Parquet)
# Este es el archivo vital que leerá tu Streamlit
ruta_app = f"s3a://{bucket_name}/gold/app_inmuebles/"

print("📲 Generando archivo optimizado para la App...")

df_gold_kpi.coalesce(1).write \
    .format("parquet") \
    .mode("overwrite") \
    .option("fs.s3a.access.key", aws_access_key) \
    .option("fs.s3a.secret.key", aws_secret_key) \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .save(ruta_app)

print(f"🚀 ¡LISTO! Archivo final disponible en: {ruta_app}")

In [0]:
# --- GUARDAR PARA LA APP (GOLD DETAIL) ---
# Usamos 'coalesce(1)' para que genere UN solo archivo .parquet (más fácil de descargar)
ruta_app = f"s3a://{bucket_name}/gold/app_inmuebles/"

print("📲 Guardando datos para Streamlit...")

df_gold_kpi.coalesce(1).write \
    .format("parquet") \
    .mode("overwrite") \
    .option("fs.s3a.access.key", aws_access_key) \
    .option("fs.s3a.secret.key", aws_secret_key) \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .save(ruta_app)

print(f"✅ Archivo listo en: {ruta_app}")